In [ ]:
#importing the libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
import time
from datetime import datetime
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
import json
import os

# Device will determine whether to run the valing on GPU or CPU.
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
display(device)

## Training Dataset Import

In [ ]:
# --- 1. Configuration ---
base_path = '../../Data/Datasets/'

sensor_mapping = {
    's1': '1. Sensor 1/Training Dataset/',
    's2': '2. Sensor 2/Training Dataset/',
    's3': '3. Sensor 3/Training Dataset/',
    's4': '4. Sensor 4/Training Dataset/'
}

data_frames = {}

# --- 2. Automation Loop (Loading Only) ---
for key, sub_path in sensor_mapping.items():
    abs_folder = os.path.abspath(os.path.join(base_path, sub_path))
    
    if not os.path.exists(abs_folder):
        print(f"Directory not found: {abs_folder}")
        continue

    # Filter for data files
    files = [f for f in os.listdir(abs_folder) if f.endswith(('.txt'))]

    if files:
        full_path = os.path.join(abs_folder, files[0])
        # Load the dataset and store in dictionary
        data_frames[key] = pd.read_csv(full_path, header=0)
    else:
        print(f"No files found in: {abs_folder}")

# --- 3. Extract to variables ---
data_s1 = data_frames.get('s1')
data_s2 = data_frames.get('s2')
data_s3 = data_frames.get('s3')
data_s4 = data_frames.get('s4')

# --- 4. Bulk Shuffling ---
datasets = [data_s1, data_s2, data_s3, data_s4]
# Shuffling each and updating the variables
data_s1, data_s2, data_s3, data_s4 = [
    df.sample(frac=1).reset_index(drop=True) if df is not None else None 
    for df in datasets
]

print("\nAll datasets loaded and shuffled.")

In [ ]:
# Validate column counts and align names
# We'll use the column names of the first dataset as the template
target_col_count = datasets[0].shape[1]
consistent_columns = True

for i, df in enumerate(datasets):
    if df.shape[1] != target_col_count:
        print(f"Error: Sensor {i+1} has {df.shape[1]} columns, but Sensor 1 has {target_col_count}.")
        consistent_columns = False
        break
    else:
        # Rename columns to match the first dataset to ensure smooth concatenation
        df.columns = datasets[0].columns

# Concatenate if all checks pass
if consistent_columns:
    final_combined_data = pd.concat(datasets, axis=0, ignore_index=True)
    print(f"Final Combined Data Shape: {final_combined_data.shape}")
else:
    print("Combination failed due to column mismatch.")

In [ ]:
input=final_combined_data.iloc[:,0:16].values
output=final_combined_data.iloc[:,16:20].values

In [ ]:
x_train = torch.Tensor(input)
y_train = torch.Tensor(output)

In [ ]:
# verify sizes
y_train.size()

In [ ]:
# Check for NaN in x_train
nan_indices_x = torch.isnan(x_train).any(dim=1)

# Remove rows with NaN from both x_train and y_train
x_train = x_train[~nan_indices_x]
y_train = y_train[~nan_indices_x]

# Check for NaN in y_train 
nan_indices_y = torch.isnan(y_train).any(dim=1)
x_train = x_train[~nan_indices_y]
y_train = y_train[~nan_indices_y]

In [ ]:
## Open saved normalization statistics (fixed)

# --- 1. Load Saved Normalization Statistics ---
with open("../../Data/Trained Models/LOGO-CV_Single_Contact_Estimation - Trained Models/normalization_statistics_fixed.json", "r") as f:
    stats = json.load(f)

# Convert to appropriate types for Torch
s_mean_in = stats["mean_input_train"]
s_std_in = stats["std_input_train"]
s_mean_y = torch.tensor(stats["mean_y"]) 
s_std_y = torch.tensor(stats["std_y"])

In [ ]:
# Reshape to a 4x4 capacitive image
x_train = x_train.view((x_train.shape[0], 4, 4, ))

In [ ]:
# verify sizes
x_train.size()

In [ ]:
input_transform = transforms.Normalize(mean=[s_mean_in], std=[s_std_in])
x_train_normalized = input_transform(x_train)

# --- Normalize Outputs using SAVED stats ---
# We only normalize the first 3 columns (X, Y, Force)
# Column 4 (Contact/No Contact) remains as 0s and 1s
y_coords_force = y_train[:, :3]
y_contact = y_train[:, 3].unsqueeze(1)

y_normalized_coords = (y_coords_force - s_mean_y) / s_std_y

# Recombine normalized coordinates with raw contact binary
y_train_normalized = torch.cat((y_normalized_coords, y_contact), dim=1)

# --- Verification ---
print(f"Final Input Shape: {x_train_normalized.shape}")
print(f"Final Output Shape: {y_train_normalized.shape}")

In [ ]:
x_train_normalized = x_train_normalized.to(device)
y_train_normalized = y_train_normalized.to(device)

In [ ]:
# Create PyTorch datasets and data loaders
train_dataset = TensorDataset(x_train_normalized, y_train_normalized)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, drop_last=True)

## Load Model Architecture

In [ ]:
class CNNRegression(nn.Module):
    def __init__(self):
        super(CNNRegression, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(4)
        
        self.conv2 = nn.Conv2d(in_channels=4, out_channels=8, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(8)
        
        self.conv3 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(16)
        
        self.conv4 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.bn4 = nn.BatchNorm2d(32)

        self.conv5 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.bn5 = nn.BatchNorm2d(64)
        
        self.fc1 = nn.Linear(in_features=64*4*4, out_features=128)  # Assuming the tensor size after conv5
        self.fc2 = nn.Linear(in_features=128, out_features=4)
        
    def forward(self, x):
        x = x.unsqueeze(1)
        
        x = self.conv1(x)
        x = self.bn1(x)
        x = nn.functional.relu(x)
        
        x = self.conv2(x)
        x = self.bn2(x)
        x = nn.functional.relu(x)
        
        x = self.conv3(x)
        x = self.bn3(x)
        x = nn.functional.relu(x)
        
        x = self.conv4(x)
        x = self.bn4(x)
        x = nn.functional.relu(x)

        x = self.conv5(x)
        x = self.bn5(x)
        x = nn.functional.relu(x)
        
        x = x.view(x.size(0), -1)
        
        x = self.fc1(x)
        x = nn.functional.relu(x)
        
        x = self.fc2(x)
        
        return x

In [ ]:
# Instantiate the model
model = CNNRegression()

# Generate a random 4x4 input tensor
input_tensor = torch.rand(4, 4)

# Run one iteration
output_tensor = model(input_tensor.unsqueeze(0))  # Unsqueezing to add batch dimension

# Print the output tensor
print("Output tensor size:", output_tensor.size())
print("Output tensor:", output_tensor)

In [ ]:
model = CNNRegression()
model = model.to(device)

In [ ]:
model.cuda

In [ ]:
data_parallel_model = torch.nn.DataParallel(model)

## Run the training

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(data_parallel_model.parameters(), lr=0.001, weight_decay=1e-5)
reg_mse_loss = nn.MSELoss()
class_bce_loss = nn.BCEWithLogitsLoss()
reg_weight_location = 0.3
reg_weight_force = 0.5
reg_weight_classification = 1 - reg_weight_location - reg_weight_force
num_epochs = 3

# Lists to store epoch-wise unnormalized MAE
unnorm_mae_x = []
unnorm_mae_y = []
unnorm_mae_force = []

s_mean_y = s_mean_y.to(device)
s_std_y = s_std_y.to(device)

for epoch in range(num_epochs):
    epoch_x_err, epoch_y_err, epoch_force_err = 0, 0, 0
    
    data_parallel_model.train()
    for i, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = data_parallel_model(inputs)
        
        # --- Standard Loss Calculation (on normalized data) ---
        loss_reg_loc = reg_mse_loss(outputs[:, 0:2], targets[:, 0:2])
        loss_reg_force = reg_mse_loss(outputs[:, 2:3], targets[:, 2:3])
        loss_class = class_bce_loss(outputs[:, 3:4], targets[:, 3:4])
        
        combined_loss = (reg_weight_location * loss_reg_loc + reg_weight_force * loss_reg_force + reg_weight_classification * loss_class)
        
        combined_loss.backward()
        optimizer.step()

        # --- Unnormalization for Plotting (Physical Units) ---
        with torch.no_grad():
            # Denormalize: (val * std) + mean
            unnorm_out = (outputs[:, 0:3] * s_std_y) + s_mean_y
            unnorm_tar = (targets[:, 0:3] * s_std_y) + s_mean_y
            
            # Calculate Absolute Error per component
            errors = torch.abs(unnorm_out - unnorm_tar)
            
            epoch_x_err += errors[:, 0].mean().item()
            epoch_y_err += errors[:, 1].mean().item()
            epoch_force_err += errors[:, 2].mean().item()

    # Average the unnormalized error for the epoch
    unnorm_mae_x.append(epoch_x_err / len(train_loader))
    unnorm_mae_y.append(epoch_y_err / len(train_loader))
    unnorm_mae_force.append(epoch_force_err / len(train_loader))

    print(f"Epoch [{epoch+1}/{num_epochs}] Finished. "
          f"MAE: X={unnorm_mae_x[-1]:.2f}mm, Y={unnorm_mae_y[-1]:.2f}mm, F={unnorm_mae_force[-1]:.2f}N")

In [ ]:
# Saving model 
torch.save(data_parallel_model.module.state_dict(),'../../Data/Trained Models/LOGO-CV_Single_Contact_Estimation - Trained Model/Model_SPCE_S1-S4.pt')
time.sleep(5)